# 🛡️ Live Session Activity: Responsible AI Governance Stack
## Build a Production-Grade Privacy Architecture for an Indian Healthcare Agent

---

**Duration:** 45–50 minutes | **Groups:** 3–4 participants | **Agentic AI Course — Responsible AI & Ethics Session**

---

### The Bridge from This Session

The session demo built a **Privacy Layer** with two components:

```python
# SESSION DEMO — Privacy Layer Implementation
#
# mask_private_data(text):       regex masking for phone, email, card, PAN, Aadhaar
# sanitize_and_respond(input):   call mask_private_data() → call LLM
#
# What the demo covers:
#   ✅ Masking PII in user INPUT before it reaches the LLM
#   ✅ Indian identifiers: PAN, Aadhaar, phone numbers
#   ✅ Privacy-by-design as a pre-processing middleware step
#
# What the demo deliberately leaves out:
#   ❌ Output scanning  — LLMs can GENERATE PII in their responses
#   ❌ Audit trail      — no record of what was masked, when, or for whom
#   ❌ Role-based policy — doctor and patient get same masking rules
#   ❌ Risk classification — binary mask/pass; no triage for BLOCK vs SANITIZE vs WARN
```

**Today you build the missing three layers** — turning the session's single-function middleware into a production-grade **Responsible AI Governance Stack** for a regulated healthcare context.

---

### Scenario: MediGuard AI — Bengaluru Healthcare Startup

MediGuard AI provides an AI-powered telemedicine assistant to patients, doctors, and hospital administrators across tier-1 and tier-2 Indian cities. Patients ask health queries, share symptoms, and sometimes inadvertently include Aadhaar numbers, phone numbers, and financial data in their messages.

The session demo's `sanitize_and_respond()` is already deployed as the input layer. But MediGuard AI has just received a DPDP (Digital Personal Data Protection Act 2023) compliance audit notice. The auditors found three gaps:

1. **LLM responses** sometimes echo back patient identifiers — no output scanning exists  
2. **No audit log** — the company cannot prove what data was masked and when  
3. **Doctors and admins** are getting the same masking as patients — violating role-based access control
4. **High-risk queries** (e.g., someone sharing full Aadhaar + bank account + diagnosis) are processed the same as low-risk ones — no triage

```
Patient/Doctor Query
        ↓
┌─────────────────────────────────────────┐
│  Group C — Role-Based Policy Engine     │  "What rules apply to THIS user?"
│  Selects masking policy by user role    │
└─────────────┬───────────────────────────┘
              ↓
┌─────────────────────────────────────────┐
│  SESSION DEMO — mask_private_data()     │  (already built — PII masking)
│  Applies role-specific patterns         │
└─────────────┬───────────────────────────┘
              ↓
┌─────────────────────────────────────────┐
│  Group D — Risk Classifier              │  "Is this safe to process?"
│  Score → BLOCK / SANITIZE / WARN / OK  │
└─────────────┬───────────────────────────┘
              ↓
           LLM call
              ↓
┌─────────────────────────────────────────┐
│  Group A — Output Scanner               │  "Did the LLM leak any PII back?"
│  Detects & redacts PII in responses     │
└─────────────┬───────────────────────────┘
              ↓
┌─────────────────────────────────────────┐
│  Group B — Audit Trail Logger           │  "Prove what happened for compliance"
│  Immutable event log + DPDP report      │
└─────────────┬───────────────────────────┘
              ↓
        Final Response
```

### Phase Timeline

| Phase | Task | Time |
|-------|------|----- |
| 0 | Setup — install packages, run the session demo baseline | 5 min |
| 1 | Session bridge + group assignments | 8 min |
| 2 | Build your governance component | 15 min |
| 3 | Integrate the full pipeline and run compliance test scenarios | 12 min |
| 4 | Reflect — DPDP audit simulation | 8 min |

---
## Phase 0: Setup (5 min)

In [1]:
#!pip install langchain-openai --quiet

import re
import json
import uuid
from datetime import datetime
from typing import Dict, List, Tuple, Optional
from langchain_openai import ChatOpenAI

print("✅ Packages ready")

✅ Packages ready


In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
import os
#os.environ["OPENAI_API_KEY"] = "PASTE_YOUR_KEY_HERE"

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print("✅ LLM ready")

✅ LLM ready


In [5]:
# ============================================================
# SESSION DEMO CODE — Reproduced Exactly
# DO NOT MODIFY — this is the baseline your group extends
# ============================================================

def mask_private_data(text: str) -> str:
    """
    SESSION DEMO function — detects and masks PII using regex.
    Identical to the Privacy Layer Implementation demo.
    """
    # Phone numbers (India + international)
    text = re.sub(
        r'(?:(?:\+?91[\s\-]?)?(?:0)?)(?:[6-9]\d{9})\b|(?<!\d)\d{3}[\s\-]?\d{3}[\s\-]?\d{4}\b',
        "[PHONE]", text)
    # Email addresses
    text = re.sub(
        r'[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}',
        "[EMAIL]", text, flags=re.IGNORECASE)
    # Credit card numbers (13–19 digits)
    text = re.sub(r'\b(?:\d[ -]*?){13,19}\b', "[CARD]", text)
    # Indian PAN format
    text = re.sub(r'\b[A-Z]{5}\d{4}[A-Z]\b', "[PAN]", text)
    # Aadhaar-like 12-digit sequences
    text = re.sub(r'\b\d{12}\b', "[AADHAAR]", text)
    return text


def sanitize_and_respond(user_input: str) -> str:
    """
    SESSION DEMO function — sanitizes input, then calls LLM.
    """
    sanitized = mask_private_data(user_input)
    response = llm.invoke(sanitized)
    return f"Sanitized Input:\n{sanitized}\n\nModel Response:\n{response.content}"


# Quick test to confirm the session baseline works
test = mask_private_data("My Aadhaar is 999988887777 and phone is 9876543210")
print("Session demo baseline test:")
print(f"  Input:  My Aadhaar is 999988887777 and phone is 9876543210")
print(f"  Output: {test}")
print("\n✅ Session baseline ready. Your group builds ON TOP of this.")

Session demo baseline test:
  Input:  My Aadhaar is 999988887777 and phone is 9876543210
  Output: My Aadhaar is 99[PHONE] and phone is [PHONE]

✅ Session baseline ready. Your group builds ON TOP of this.


---
## Phase 1: Session Bridge + Group Assignments (8 min)

### What the Session Demo Does vs What You're Building

```
SESSION DEMO:          User Input → mask_private_data() → LLM → response

TODAY (your stack):    User Input → PolicyEngine (C) → mask_private_data() 
                                 → RiskClassifier (D) → LLM 
                                 → OutputScanner (A) → AuditLogger (B) → response
```

### Group Assignments

| Group | Component | The Gap It Fixes | DPDP Requirement |
|-------|-----------|-----------------|------------------|
| **A** | Output Privacy Scanner | Session masks input. LLMs can generate PII in output. | Article 8: Data minimisation in all processing |
| **B** | Audit Trail Logger | Session has no record of masking events. | Article 12: Data principal rights & accountability |
| **C** | Role-Based Policy Engine | Same rules for doctors and patients — violates need-to-know | Article 6: Consent & purpose limitation |
| **D** | PII Risk Classifier | Binary mask/pass — no triage for high-risk queries | Article 9: Special category data (health data) |

---
## Phase 2: Build Your Component (15 min)

In [7]:
# ============================================================
# GROUP A — Output Privacy Scanner
# ============================================================
# PROBLEM: The session demo masks user INPUT before calling the LLM.
# But the LLM can still generate PII in its RESPONSE — for example:
#   - Echoing a phone number the user mentioned
#   - Generating a realistic-looking Aadhaar in an example
#   - Suggesting the user contact someone at a specific email
#
# Your task: build a scanner that runs on the LLM's OUTPUT (not input)
# and detects any PII that leaked through.
#
# TASKS:
#   1. Complete scan_llm_output() — detect PII tokens in the response text
#   2. Complete redact_output()   — replace detected PII with [REDACTED-TYPE]
#   3. Complete OutputScanner.process() — decide: redact, warn, or pass
#   4. Test with the sample responses below
#
# HINT: You can reuse mask_private_data() from the session demo,
#       but your scanner must also DETECT (not just replace) —
#       it needs to return WHAT was found and HOW MANY items.

class OutputScanner:
    """
    Scans LLM output for PII leakage.
    Extends the session demo's input masking to cover the output side.
    """

    # PII patterns — same identifiers as session demo, but now on output
    PATTERNS = {
        "PHONE":   r'(?:(?:\+?91[\s\-]?)?(?:0)?)(?:[6-9]\d{9})\b|(?<!\d)\d{3}[\s\-]?\d{3}[\s\-]?\d{4}\b',
        "EMAIL":   r'[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}',
        "CARD":    r'\b(?:\d[ -]*?){13,19}\b',
        "PAN":     r'\b[A-Z]{5}\d{4}[A-Z]\b',
        "AADHAAR": r'\b\d{12}\b',
    }

    def scan_llm_output(self, response_text: str) -> Dict[str, List[str]]:
        """
        Scans the LLM response for PII tokens.
        Returns a dict mapping PII type → list of matched values found.

        Example output:
          {"PHONE": ["9876543210"], "EMAIL": ["doc@hospital.in"]}

        IMPLEMENT THIS:
        - Loop over self.PATTERNS
        - Use re.findall() to collect all matches
        - Return only non-empty entries
        """
        # TODO: implement
        found = {}
        # YOUR CODE HERE
        return found

    def redact_output(self, response_text: str) -> str:
        """
        Replaces any PII found in the response with [REDACTED-TYPE] tokens.
        Returns the cleaned response text.

        IMPLEMENT THIS:
        - Loop over self.PATTERNS
        - Use re.sub() to replace with "[REDACTED-{pii_type}]"
        """
        # TODO: implement
        cleaned = response_text
        # YOUR CODE HERE
        return cleaned

    def process(self, response_text: str) -> Dict:
        """
        Main entry point. Scans and redacts output.
        Returns:
          {
            "original_length": int,
            "pii_found": dict of matches,
            "pii_count": total number of PII instances,
            "redacted_output": cleaned text,
            "leakage_detected": bool
          }

        IMPLEMENT THIS:
        - Call scan_llm_output() to find PII
        - Call redact_output() to clean the text
        - Count total PII instances
        - Set leakage_detected = True if any PII found
        """
        # TODO: implement
        pass


# ── Test your OutputScanner ─────────────────────────────────────────
# These simulate LLM responses that accidentally contain PII
scanner = OutputScanner()

test_responses = [
    # Clean response — should pass with no leakage
    "Based on your symptoms, I recommend consulting a cardiologist. Please schedule an appointment.",
    
    # Leaky response — LLM echoed back a phone number
    "I understand you have concerns. You can reach Dr. Sharma at 9876543210 for follow-up.",
    
    # Leaky response — LLM generated a fake Aadhaar in an example
    "For example, an Aadhaar number looks like 234500123456. Never share this publicly.",
]

print("Group A — OutputScanner Test")
print("=" * 50)
for i, resp in enumerate(test_responses, 1):
    result = scanner.process(resp)
    if result:  # check implementation exists
        status = "⚠️  LEAKAGE DETECTED" if result.get("leakage_detected") else "✅ Clean"
        print(f"\nTest {i}: {status}")
        print(f"  PII found: {result.get('pii_found', {})}")
        print(f"  Redacted: {result.get('redacted_output', '')[:100]}")
    else:
        print(f"\nTest {i}: (implement process() to see results)")

Group A — OutputScanner Test

Test 1: (implement process() to see results)

Test 2: (implement process() to see results)

Test 3: (implement process() to see results)


In [ ]:
# ============================================================
# GROUP B — Audit Trail Logger
# ============================================================
# PROBLEM: The session demo masks data but keeps ZERO record of it.
# Under DPDP 2023, a healthcare company must be able to prove:
#   - When was personal data processed?
#   - What type of data was involved?
#   - Who (which user/session) submitted it?
#   - What action was taken on it?
#
# Your task: build an immutable event log and a compliance report generator.
#
# TASKS:
#   1. Complete AuditEvent dataclass (or dict structure)
#   2. Complete AuditLogger.log_masking_event() — record what was masked
#   3. Complete AuditLogger.log_output_scan()  — record output scan result
#   4. Complete AuditLogger.generate_compliance_report() — DPDP-style summary
#   5. Test by running several sessions and printing the report

class AuditLogger:
    """
    Immutable event log for all privacy-related actions.
    Provides the audit trail required by DPDP 2023 Article 12.
    """

    def __init__(self):
        self._events: List[Dict] = []   # append-only, never delete

    def log_masking_event(
        self,
        session_id: str,
        user_role: str,
        original_text: str,
        sanitized_text: str,
        pii_types_detected: List[str]
    ) -> str:
        """
        Records a PII masking event.
        Returns the event_id (UUID) for cross-referencing.

        IMPLEMENT THIS — create a dict with:
          event_id:          str(uuid.uuid4())
          timestamp:         datetime.utcnow().isoformat() + 'Z'
          event_type:        'INPUT_MASKING'
          session_id:        session_id
          user_role:         user_role
          pii_types:         pii_types_detected  (list of strings)
          pii_count:         len(pii_types_detected)
          original_length:   len(original_text)
          sanitized_length:  len(sanitized_text)
          data_reduced:      original_length > sanitized_length

        Append to self._events and return event_id.
        """
        # TODO: implement
        pass

    def log_output_scan(
        self,
        session_id: str,
        leakage_detected: bool,
        pii_types_leaked: List[str],
        action_taken: str   # 'REDACTED' or 'PASSED'
    ) -> str:
        """
        Records an output scan event.
        Returns the event_id.

        IMPLEMENT THIS — create a dict with:
          event_id:          str(uuid.uuid4())
          timestamp:         datetime.utcnow().isoformat() + 'Z'
          event_type:        'OUTPUT_SCAN'
          session_id:        session_id
          leakage_detected:  leakage_detected
          pii_types_leaked:  pii_types_leaked
          action_taken:      action_taken
        """
        # TODO: implement
        pass

    def generate_compliance_report(self) -> Dict:
        """
        Generates a DPDP-style compliance summary report.
        Returns a structured dict suitable for a data protection officer.

        IMPLEMENT THIS — the report should include:
          report_generated_at:     datetime.utcnow().isoformat() + 'Z'
          total_events:            len(self._events)
          masking_events:          count of INPUT_MASKING events
          output_scan_events:      count of OUTPUT_SCAN events
          leakage_incidents:       count where leakage_detected == True
          pii_types_encountered:   list of unique PII types across all events
          sessions_processed:      count of unique session_ids
          dpdp_compliant:          True if leakage_incidents == 0

        BONUS: add a 'top_pii_types' field showing most frequent PII types
        """
        # TODO: implement
        pass

    def get_events(self) -> List[Dict]:
        """Returns all logged events (read-only copy)."""
        return list(self._events)


# ── Test your AuditLogger ────────────────────────────────────────────
audit = AuditLogger()

# Simulate several masking events
test_sessions = [
    ("sess-001", "patient", "My Aadhaar is 999988887777, phone 9876543210",
     "My Aadhaar is [AADHAAR], phone [PHONE]", ["AADHAAR", "PHONE"]),
    ("sess-002", "doctor", "Patient email dr@hospital.in needs callback",
     "Patient email [EMAIL] needs callback", ["EMAIL"]),
    ("sess-003", "patient", "Card 1234-5678-9876-5432 and PAN ABCDE1234F",
     "Card [CARD] and PAN [PAN]", ["CARD", "PAN"]),
]

print("Group B — AuditLogger Test")
print("=" * 50)

for sess_id, role, orig, san, pii_types in test_sessions:
    eid = audit.log_masking_event(sess_id, role, orig, san, pii_types)
    if eid:
        print(f"  ✅ Logged event {eid[:8]}... for session {sess_id} [{role}]")
    else:
        print(f"  (implement log_masking_event() to see results)")

# Log a simulated output leakage
audit.log_output_scan("sess-002", leakage_detected=True,
                       pii_types_leaked=["PHONE"], action_taken="REDACTED")

# Print compliance report
print("\nCompliance Report:")
report = audit.generate_compliance_report()
if report:
    for k, v in report.items():
        print(f"  {k}: {v}")
else:
    print("  (implement generate_compliance_report() to see results)")

In [ ]:
# ============================================================
# GROUP C — Role-Based Policy Engine
# ============================================================
# PROBLEM: The session demo applies IDENTICAL masking to every user.
# In a healthcare system, different roles have different data needs:
#
#   PATIENT:  Cannot see other patients' data. Aadhaar always masked.
#             Phone/email masked if not their own session.
#
#   DOCTOR:   Can see patient phone numbers (needed for callbacks).
#             Cannot see financial data (PAN, card numbers).
#             Aadhaar still masked.
#
#   ADMIN:    Can see all contact info for operational tasks.
#             Aadhaar still masked (legally protected always).
#             PAN visible for billing/insurance.
#
# Your task: build a PolicyEngine that selects the right masking
# rules per role and returns role-specific sanitized text.
#
# TASKS:
#   1. Define ROLE_POLICIES — dict mapping role → which PII types to mask
#   2. Complete PolicyEngine.get_policy() — return policy for a role
#   3. Complete PolicyEngine.apply_policy() — mask only the PII types
#                                             that this role should NOT see
#   4. Complete PolicyEngine.explain_policy() — human-readable policy summary
#   5. Test all three roles on the same input and compare outputs

# ── Role Policy Definitions ─────────────────────────────────────────
# Each role specifies which PII types should be MASKED for that role.
# AADHAAR is always masked for all roles — it's legally protected under DPDP.

ROLE_POLICIES = {
    "patient": {
        "description": "Patient — maximum privacy, only sees anonymised data",
        "mask": ["PHONE", "EMAIL", "CARD", "PAN", "AADHAAR"],   # mask everything
        "allowed": []   # no PII visible
    },
    "doctor": {
        "description": "Doctor — can see contact info for clinical callbacks",
        "mask": ["CARD", "PAN", "AADHAAR"],   # mask financial + identity
        "allowed": ["PHONE", "EMAIL"]          # can see contact info
    },
    "admin": {
        "description": "Admin — operational access to billing and contact data",
        "mask": ["AADHAAR"],                    # only Aadhaar always masked
        "allowed": ["PHONE", "EMAIL", "CARD", "PAN"]
    }
}


class PolicyEngine:
    """
    Role-based privacy policy engine.
    Extends the session demo's uniform masking to per-role rules.
    """

    PII_PATTERNS = {
        "PHONE":   r'(?:(?:\+?91[\s\-]?)?(?:0)?)(?:[6-9]\d{9})\b|(?<!\d)\d{3}[\s\-]?\d{3}[\s\-]?\d{4}\b',
        "EMAIL":   r'[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}',
        "CARD":    r'\b(?:\d[ -]*?){13,19}\b',
        "PAN":     r'\b[A-Z]{5}\d{4}[A-Z]\b',
        "AADHAAR": r'\b\d{12}\b',
    }

    def get_policy(self, role: str) -> Dict:
        """
        Returns the masking policy for a given role.
        Defaults to 'patient' policy if role is unrecognised.

        IMPLEMENT THIS:
        - Look up ROLE_POLICIES[role.lower()]
        - If role not found, return ROLE_POLICIES['patient'] (fail-safe default)
        """
        # TODO: implement
        pass

    def apply_policy(self, text: str, role: str) -> Tuple[str, List[str]]:
        """
        Applies role-specific masking to the input text.
        Returns (masked_text, list_of_pii_types_that_were_masked).

        IMPLEMENT THIS:
        - Get policy for this role
        - Only apply regex masking for PII types in policy['mask']
        - Track which types were actually masked (had matches)
        - Return masked text and list of masked PII types

        Example:
          apply_policy("phone 9876543210 and PAN ABCDE1234F", "doctor")
          # Doctor policy masks CARD, PAN, AADHAAR but NOT PHONE
          → ("phone 9876543210 and PAN [PAN]", ["PAN"])
        """
        # TODO: implement
        pass

    def explain_policy(self, role: str) -> str:
        """
        Returns a plain-English explanation of what a role can and cannot see.

        IMPLEMENT THIS:
        - Get policy for role
        - Format as: "Role: {role}\n  ✅ Can see: {allowed}\n  🔒 Always masked: {mask}"
        """
        # TODO: implement
        pass


# ── Test your PolicyEngine ───────────────────────────────────────────
engine = PolicyEngine()

# Same message sent by patient, viewed by doctor, processed by admin
test_message = (
    "Patient Priya, Aadhaar 999988887777, phone 9876543210, "
    "email priya@email.com, card 4111111111111111, PAN ABCDE1234F"
)

print("Group C — PolicyEngine Test")
print("=" * 50)
print(f"Original: {test_message}\n")

for role in ["patient", "doctor", "admin"]:
    result = engine.apply_policy(test_message, role)
    explanation = engine.explain_policy(role)
    if result:
        masked_text, masked_types = result
        print(f"Role: {role.upper()}")
        print(f"  Policy: {explanation}")
        print(f"  Masked types: {masked_types}")
        print(f"  Result: {masked_text}")
        print()
    else:
        print(f"Role: {role.upper()} — (implement apply_policy() to see results)\n")

print("❓ Discussion: Why must AADHAAR always be masked regardless of role?")
print("   Hint: DPDP 2023 Section 16 — children and sensitive identifiers")

In [ ]:
# ============================================================
# GROUP D — PII Risk Classifier
# ============================================================
# PROBLEM: The session demo is binary — either mask or don't mask.
# In production, risk severity matters:
#
#   A message with ONE phone number → LOW risk → mask and proceed
#   A message with Aadhaar + PAN + card → CRITICAL risk → BLOCK or human review
#   A message with health keywords + Aadhaar → HIGH risk → mandatory consent check
#
# Your task: build a risk classifier that scores messages and returns
# an action: ALLOW / SANITIZE / WARN / BLOCK
#
# TASKS:
#   1. Define PII_RISK_WEIGHTS  — risk score per PII type (0–10 scale)
#   2. Complete PIIRiskClassifier.detect_pii_types() — list PII in text
#   3. Complete PIIRiskClassifier.score()   — compute total risk score
#   4. Complete PIIRiskClassifier.classify() — map score to action
#   5. Complete PIIRiskClassifier.triage()   — full decision with reasoning
#   6. Test with the 4 sample messages below

# Risk weight per PII type — higher = more sensitive
# These reflect DPDP 2023's definition of sensitive personal data
PII_RISK_WEIGHTS = {
    "PHONE":    2,   # Contact info — low sensitivity
    "EMAIL":    2,   # Contact info — low sensitivity
    "CARD":     6,   # Financial data — high sensitivity
    "PAN":      5,   # Tax identity — medium-high sensitivity
    "AADHAAR": 9,   # Biometric-linked national ID — critical
}

# Decision thresholds
RISK_THRESHOLDS = {
    "ALLOW":    (0, 2),    # score 0–2:  safe to proceed as-is
    "SANITIZE": (3, 7),    # score 3–7:  mask and proceed
    "WARN":     (8, 12),   # score 8–12: flag for human review, proceed masked
    "BLOCK":    (13, 100), # score 13+:  refuse to process, escalate
}

# Health-related keywords that escalate risk (special category data)
HEALTH_KEYWORDS = [
    "diagnosis", "disease", "medication", "prescription", "surgery",
    "blood", "hiv", "cancer", "diabetes", "mental health", "psychiatric"
]


class PIIRiskClassifier:
    """
    Risk-based PII triage for the governance stack.
    Extends the session demo's binary mask/pass to a 4-level action system.
    """

    PII_PATTERNS = {
        "PHONE":   r'(?:(?:\+?91[\s\-]?)?(?:0)?)(?:[6-9]\d{9})\b|(?<!\d)\d{3}[\s\-]?\d{3}[\s\-]?\d{4}\b',
        "EMAIL":   r'[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}',
        "CARD":    r'\b(?:\d[ -]*?){13,19}\b',
        "PAN":     r'\b[A-Z]{5}\d{4}[A-Z]\b',
        "AADHAAR": r'\b\d{12}\b',
    }

    def detect_pii_types(self, text: str) -> List[str]:
        """
        Returns list of PII types present in text (no duplicates).
        E.g. "phone 9876543210 and aadhaar 999988887777" → ["PHONE", "AADHAAR"]

        IMPLEMENT THIS:
        - Loop over self.PII_PATTERNS
        - If re.search() finds a match, add the PII type to the result list
        """
        # TODO: implement
        pass

    def score(self, text: str) -> Tuple[int, List[str], bool]:
        """
        Calculates a risk score from 0 to ~100.
        Returns (risk_score, pii_types_found, health_data_present).

        Scoring rules:
          base_score = sum of PII_RISK_WEIGHTS for each detected PII type
          health_multiplier: if any HEALTH_KEYWORDS in text → add +4 bonus
          multiple_pii_penalty: if 3+ PII types → add +3 bonus

        IMPLEMENT THIS:
        - Detect PII types with detect_pii_types()
        - Sum up their weights from PII_RISK_WEIGHTS
        - Check for health keywords
        - Apply bonuses
        - Return (total_score, pii_types, health_present)
        """
        # TODO: implement
        pass

    def classify(self, risk_score: int) -> str:
        """
        Maps a risk score to an action: 'ALLOW' / 'SANITIZE' / 'WARN' / 'BLOCK'

        IMPLEMENT THIS:
        - Loop over RISK_THRESHOLDS
        - Return the action whose (min, max) range contains risk_score
        """
        # TODO: implement
        pass

    def triage(self, text: str) -> Dict:
        """
        Full triage decision for a message.
        Returns a structured result dict.

        IMPLEMENT THIS — return dict with:
          risk_score:       int
          action:           'ALLOW' / 'SANITIZE' / 'WARN' / 'BLOCK'
          pii_types_found:  list of PII types
          health_data:      bool
          reason:           plain-English explanation of the decision
          proceed:          True if action is NOT 'BLOCK'

        BONUS: Add a 'recommended_role_minimum' field — what is the minimum
        role that should handle this message? (patient, doctor, admin)
        """
        # TODO: implement
        pass


# ── Test your PIIRiskClassifier ──────────────────────────────────────
classifier = PIIRiskClassifier()

triage_cases = [
    # ALLOW — plain question, no PII
    "What are the symptoms of dehydration?",
    
    # SANITIZE — one phone number, low risk
    "Please call me at 9876543210 to confirm my appointment.",
    
    # WARN — Aadhaar present, health context
    "My Aadhaar 999988887777 is linked to my diabetes medication records.",
    
    # BLOCK — Aadhaar + PAN + card + health keywords
    "Aadhaar 999988887777, PAN ABCDE1234F, card 4111-1111-1111-1111, "
    "HIV diagnosis attached to my insurance claim.",
]

print("Group D — PIIRiskClassifier Test")
print("=" * 50)

for i, msg in enumerate(triage_cases, 1):
    result = classifier.triage(msg)
    if result:
        action_icons = {"ALLOW": "✅", "SANITIZE": "🔧", "WARN": "⚠️", "BLOCK": "🚫"}
        icon = action_icons.get(result.get("action", ""), "?")
        print(f"\nCase {i}: {icon} {result.get('action')} (score: {result.get('risk_score')})")
        print(f"  PII types: {result.get('pii_types_found')}")
        print(f"  Health data: {result.get('health_data')}")
        print(f"  Reason: {result.get('reason', '')[:120]}")
        print(f"  Proceed: {result.get('proceed')}")
    else:
        print(f"\nCase {i}: (implement triage() to see results)")

print("\n❓ Discussion: Is 'BLOCK' the right action, or should you SANITIZE + WARN?")
print("   In healthcare, blocking can deny care. How do you balance privacy vs access?")

---
## Phase 3: Integrate the Full Pipeline (12 min)

In [ ]:
# ── Stubs for incomplete groups ──────────────────────────────────────
# If a group hasn't finished, this stub replaces their component
# so the full pipeline still runs.

class _StubOutputScanner:
    def process(self, text):
        return {"pii_found": {}, "pii_count": 0, "redacted_output": text,
                "leakage_detected": False, "original_length": len(text)}

class _StubAuditLogger:
    _events = []
    def log_masking_event(self, *a, **kw): return "stub-" + str(uuid.uuid4())[:8]
    def log_output_scan(self, *a, **kw): return "stub-" + str(uuid.uuid4())[:8]
    def generate_compliance_report(self):
        return {"note": "Stub active — Group B not yet implemented",
                "total_events": 0, "dpdp_compliant": None}

class _StubPolicyEngine:
    def apply_policy(self, text, role):
        return mask_private_data(text), ["PHONE","EMAIL","CARD","PAN","AADHAAR"]
    def explain_policy(self, role): return f"Stub policy for {role}"

class _StubRiskClassifier:
    def triage(self, text):
        return {"risk_score": 0, "action": "SANITIZE", "pii_types_found": [],
                "health_data": False, "proceed": True, "reason": "Stub — always SANITIZE"}

# Use real implementations if available, stubs otherwise
try:
    _scanner = OutputScanner()
    _scanner.process("test")
    output_scanner = _scanner
    print("✅ OutputScanner (Group A): real")
except: output_scanner = _StubOutputScanner(); print("⚠️  OutputScanner: stub")

try:
    _al = AuditLogger()
    _al.log_masking_event("x","y","a","b",["PHONE"])
    audit_logger = _al
    print("✅ AuditLogger   (Group B): real")
except: audit_logger = _StubAuditLogger(); print("⚠️  AuditLogger: stub")

try:
    _pe = PolicyEngine()
    _pe.apply_policy("test 9876543210", "patient")
    policy_engine = _pe
    print("✅ PolicyEngine  (Group C): real")
except: policy_engine = _StubPolicyEngine(); print("⚠️  PolicyEngine: stub")

try:
    _rc = PIIRiskClassifier()
    _rc.triage("test")
    risk_classifier = _rc
    print("✅ RiskClassifier(Group D): real")
except: risk_classifier = _StubRiskClassifier(); print("⚠️  RiskClassifier: stub")

In [ ]:
# ── Full Governance Pipeline ─────────────────────────────────────────

def governance_pipeline(
    user_input: str,
    role: str,
    session_id: str = None
) -> Dict:
    """
    Full 4-layer responsible AI governance stack.
    Extends the session demo's single-step sanitize_and_respond().
    """
    if session_id is None:
        session_id = str(uuid.uuid4())[:8]

    print(f"\n{'─'*55}")
    print(f"Session: {session_id} | Role: {role.upper()}")
    print(f"Input: {user_input[:80]}")
    print('─'*55)

    # ── STEP 1: Role-Based Policy (Group C) ────────────────────────
    policy_result = policy_engine.apply_policy(user_input, role)
    if policy_result:
        sanitized_input, masked_types = policy_result
    else:
        sanitized_input = mask_private_data(user_input)  # fallback to session demo
        masked_types = []
    print(f"[C] PolicyEngine  → masked: {masked_types}")

    # ── STEP 2: Risk Classification (Group D) ──────────────────────
    triage_result = risk_classifier.triage(user_input)  # score on original
    action = triage_result.get("action", "SANITIZE")
    print(f"[D] RiskClassifier→ score={triage_result.get('risk_score',0)}, action={action}")

    if action == "BLOCK":
        print("🚫 BLOCKED — message contains critical PII combination")
        audit_logger.log_masking_event(
            session_id, role, user_input, sanitized_input, masked_types)
        return {"status": "BLOCKED", "reason": triage_result.get("reason"),
                "response": None, "session_id": session_id}

    # ── STEP 3: Audit — Log Masking Event (Group B) ────────────────
    event_id = audit_logger.log_masking_event(
        session_id, role, user_input, sanitized_input, masked_types)
    print(f"[B] AuditLogger   → event logged: {str(event_id)[:12]}")

    # ── STEP 4: LLM Call (Session Demo pattern) ────────────────────
    if action == "WARN":
        system_note = (
            "[COMPLIANCE NOTE: This query was flagged as high-risk PII content. "
            "Respond with extra care and do not repeat any identifiers.] "
        )
        llm_input = system_note + sanitized_input
    else:
        llm_input = sanitized_input

    llm_response = llm.invoke(llm_input)
    raw_output = llm_response.content
    print(f"[LLM] Response length: {len(raw_output)} chars")

    # ── STEP 5: Output Scanning (Group A) ──────────────────────────
    scan_result = output_scanner.process(raw_output)
    final_output = scan_result.get("redacted_output", raw_output)
    leakage = scan_result.get("leakage_detected", False)

    if leakage:
        print(f"[A] OutputScanner → ⚠️  PII leakage detected! Redacted: {scan_result.get('pii_found')}")
        audit_logger.log_output_scan(
            session_id,
            leakage_detected=True,
            pii_types_leaked=list(scan_result.get("pii_found", {}).keys()),
            action_taken="REDACTED"
        )
    else:
        print(f"[A] OutputScanner → ✅ Clean output")
        audit_logger.log_output_scan(session_id, False, [], "PASSED")

    print(f"\nFinal response: {final_output[:120]}...")

    return {
        "status": "OK",
        "session_id": session_id,
        "role": role,
        "action": action,
        "masked_pii_types": masked_types,
        "leakage_detected": leakage,
        "response": final_output,
        "risk_score": triage_result.get("risk_score", 0)
    }

print("✅ Full governance pipeline defined")

In [ ]:
# ── DPDP Compliance Test Scenarios ───────────────────────────────────
# Run all 5 scenarios and observe how each group's component responds

test_scenarios = [
    # Scenario 1: Patient — safe query, no PII
    ("What are the side effects of metformin for diabetes?",
     "patient", "sess-P01"),

    # Scenario 2: Patient — shares phone, low risk
    ("Call me at 9876543210. I need advice on my blood pressure medication.",
     "patient", "sess-P02"),

    # Scenario 3: Doctor — sees phone, financial masked
    ("Patient Priya, phone 9876543210, PAN ABCDE1234F, needs follow-up on surgery.",
     "doctor", "sess-D01"),

    # Scenario 4: Patient — HIGH RISK: Aadhaar + health keyword → WARN
    ("My Aadhaar 999988887777 is linked to my HIV treatment records at AIIMS.",
     "patient", "sess-P03"),

    # Scenario 5: Patient — CRITICAL: Aadhaar + PAN + card + health → BLOCK
    ("Aadhaar 999988887777, PAN ABCDE1234F, card 4111-1111-1111-1111, "
     "please process my cancer diagnosis insurance claim.",
     "patient", "sess-P04"),
]

results = []
for query, role, sess in test_scenarios:
    result = governance_pipeline(query, role=role, session_id=sess)
    results.append(result)

print("\n" + "="*55)
print("PIPELINE SUMMARY")
print("="*55)
for r in results:
    status_icon = {"OK": "✅", "BLOCKED": "🚫"}.get(r.get("status",""), "?")
    print(f"{status_icon} {r.get('session_id')} [{r.get('role','?').upper():7}] "
          f"risk={r.get('risk_score',0):3d} action={r.get('action','?'):8s} "
          f"leak={r.get('leakage_detected',False)}")

In [ ]:
# ── Generate DPDP Compliance Report (Group B deliverable) ────────────
print("\nDPDP COMPLIANCE AUDIT REPORT")
print("=" * 55)

report = audit_logger.generate_compliance_report()
if report and "total_events" in report:
    for key, value in report.items():
        print(f"  {key:30s}: {value}")

    if report.get("dpdp_compliant") == True:
        print("\n✅ System is DPDP 2023 compliant — no leakage incidents detected")
    elif report.get("dpdp_compliant") == False:
        print("\n⚠️  COMPLIANCE ALERT — leakage incidents require remediation")
    else:
        print("\n❓ Compliance status unknown — check AuditLogger implementation")
else:
    print("  (Group B: implement generate_compliance_report() to see the DPDP report)")

---
## Phase 4: Reflect — DPDP Audit Simulation (8 min)

Imagine you are presenting this governance stack to a Data Protection Officer (DPO) during a DPDP 2023 compliance audit. Answer these questions as a group:

---

**Q1 — The Output Gap (Group A)**  
The session demo masked inputs. Your OutputScanner now covers responses too. But what about the **conversation history**? If a patient shared their Aadhaar in Message 1 and the LLM was asked to summarise the conversation in Message 5 — would your current scanner catch a leak there? What architectural change would close that gap?

_Your answer:_

---

**Q2 — The Audit Trail (Group B)**  
Your AuditLogger uses an in-memory `_events` list. The DPO asks: *"If the server crashes at 2AM, can you still prove what was processed?"* How would you make the audit trail **persistent and tamper-proof**? Name two specific technologies you'd use.

_Your answer:_

---

**Q3 — Role-Based Access (Group C)**  
Your PolicyEngine gives doctors access to phone numbers but not PAN. The DPO asks: *"What if a doctor is the data subject — their own PAN is in the message?"* Role-based policy doesn't know whose data it is, only what role is viewing. How would you add **data ownership context** to the policy decision?

_Your answer:_

---

**Q4 — Risk Thresholds (Group D)**  
Your classifier BLOCKs messages with score ≥ 13. In Scenario 5 (cancer diagnosis + Aadhaar + PAN + card), the message is blocked. But the patient genuinely needs help processing an insurance claim. Is BLOCK always the right decision for healthcare? What alternative action would balance **patient safety** against **data privacy**?

_Your answer:_

---

**Q5 — The Whole Stack (All Groups)**  
The session demo was 2 functions and 30 lines. Today's governance stack is 4 components and ~150 lines. A startup CTO says: *"This is over-engineered for our MVP."* Which **one component** from today would you argue is non-negotiable even for an MVP, and why? (There's no single right answer — defend your choice.)

_Your answer:_